In [112]:
import matplotlib.pyplot as plt
import pandas as pd
from features import (
    row_to_fen,
    preprocess_data, 
    voc_single_sample, 
    ev_board, 
    best_two_diff, 
    voc_consideration_set, 
    cp_board
)

import dask.dataframe as dd
from utils import compute_metrics_by_bin, compute_metrics_by_qbin, plot_metrics
%load_ext autoreload
%autoreload 2
DEPTH_DEEP = 14
DEPTH_SHALLOW = 1


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [122]:
n_games = 200
ddf = dd.read_parquet(f"../data/moves_{n_games}.parquet")
ddf = preprocess_data(ddf)


In [123]:
pdf = ddf.compute()

In [124]:
row = pdf.iloc[0]

In [ ]:

cp_board_shallow = lambda engine, board: cp_board(engine, board, 1)
cp_board_deep = lambda engine, board: cp_board(engine, board, 14)
best_two_diff_shallow = lambda engine, board: best_two_diff(engine, board, 1)
best_two_diff_deep = lambda engine, board: best_two_diff(engine, board, 14)

def process_partition(pdf):

    functions = [ev_board, cp_board, cp_board_shallow, best_two_diff_shallow, best_two_diff_deep, voc_single_sample, voc_consideration_set]
    engine = get_stockfish_engine()

    def process_row(row): 
        fen = row["fen"]

        board = chess.Board(fen)
        for f in functions: 
            row[f.__name__] = f(engine, board)
        return row

    
    pdf = pdf.apply(process_row, axis = 1)

    engine.quit()
    return pdf

pdf = process_partition(pdf.iloc[:10].copy())


TypeError: process_partition() missing 1 required positional argument: 'functions'

In [ ]:

# df["move_time"] = df["move_time"].fillna(0).clip(lower=0)
# move_time = df["move_time"]
# cum = df.assign(move_time=move_time).groupby(["gid", "player_white"], sort=False)["move_time"].cumsum()
# spent_prior = cum - move_time
# n_prior = df.groupby(["gid", "player_white"], sort=False).cumcount()
# inc = df["clock_increment"]
# df["player_time_left"] = (df["initial_clock"] - spent_prior + n_prior * inc).clip(lower=0)
# df["fen"] = df.apply(row_to_fen, axis = 1, meta = (str, str)) 
# df["move_number"] = (df["move_ply"] + 1) // 2
# df["total_time_left"] = df.groupby(["gid", "move_number"])["player_time_left"].transform("sum")
# df["move_time(t-1)"] = df.groupby(["gid"])["move_time"].shift(1)
# df["move_time(t-2)"] = df.groupby(["gid"])["move_time"].shift(2)
# df["fen(t-1)"] = df.groupby(["gid"])["fen"].shift(1)

Dask Series Structure:
npartitions=1
    Int32
      ...
Dask Name: groupbycumsum, 11 expressions
Expr=GroupByCumsum(frame=(Assign(frame=Assign(frame=ResetIndex(frame=SortValues(frame=ReadParquetFSSpec(6b9de21), by=['gid', 'move_ply'], options={'kind': 'mergesort'}), drop=True))))[['gid', 'move_time', 'player_white']], _slice='move_time', numeric_only=False)

,gid,board_position,move_time,move_ply,player_white,white_elo,black_elo,initial_clock,clock_increment
0,202201001000093,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,<NA>,1,True,2026,2055,600,0
1,202201001000093,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR,<NA>,2,False,2026,2055,600,0
2,202201001000093,rnbqkb1r/pppppppp/5n2/8/3P4/8/PPP1PPPP/RNBQKBNR,1,3,True,2026,2055,600,0
3,202201001000093,rnbqkb1r/pppppppp/5n2/8/2PP4/8/PP2PPPP/RNBQKBNR,1,4,False,2026,2055,600,0
4,202201001000093,rnbqkb1r/pppp1ppp/4pn2/8/2PP4/8/PP2PPPP/RNBQKBNR,2,5,True,2026,2055,600,0


In [111]:
n_games = 200
sample_games = pd.read_parquet(f"../data/moves_{n_games}.parquet")
df = preprocess_data(sample_games)

TypeError: row_to_fen() got an unexpected keyword argument 'meta'

AttributeError: 'DataFrame' object has no attribute 'to_pandas'

<dask_expr.expr.Scalar: expr=(ToSeriesIndex(frame=DropDuplicates(frame=Index(frame=AssignAlign(frame=AssignAlign(frame=AssignAlign(frame=AssignAlign(frame=Assign(frame=Assign(frame=Assign(frame=AssignAlign(frame=AssignAlign(frame=Assign(frame=SetIndex(frame=ResetIndex(frame=SortValues(frame=ReadParquetFSSpec(6b9de21), by=['gid', 'move_ply'], options={'kind': 'mergesort'}), drop=True), _other='gid', options={})), column='spent_prior', value=OpAlignPartitions(frame=GroupByCumsum(frame=(Assign(frame=SetIndex(frame=ResetIndex(frame=SortValues(frame=ReadParquetFSSpec(6b9de21), by=['gid', 'move_ply'], options={'kind': 'mergesort'}), drop=True), _other='gid', options={})))[['move_time', 'player_white']], _slice='move_time', numeric_only=False), other=(Assign(frame=SetIndex(frame=ResetIndex(frame=SortValues(frame=ReadParquetFSSpec(6b9de21), by=['gid', 'move_ply'], options={'kind': 'mergesort'}), drop=True), _other='gid', options={})))['move_time'], op='__sub__')), column='n_prior', value=Group

In [ ]:
row = df.oc[0]

NotImplementedError: 'DataFrame.iloc' only supports selecting columns. It must be used like 'df.iloc[:, column_indexer]'.

In [ ]:
engine = get_stockfish_engine()


for i, row in enumerate(df.iterrows()): 
    board = chess.Board(row["fen"])
    for f in functions: 
        row[f.__name__] = f(engine, board)
    if i > 10: break


TypeError: tuple indices must be integers or slices, not str

(0,
 gid                                               202201001000093
 board_position        rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR
 move_time                                                       0
 move_ply                                                        1
 player_white                                                 True
 white_elo                                                    2026
 black_elo                                                    2055
 initial_clock                                                 600
 clock_increment                                                 0
 player_time_left                                              600
 fen                 rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w
 move_number                                                     1
 total_time_left                                              1200
 move_time(t-1)                                               <NA>
 move_time(t-2)                                           

In [4]:
import chess
import chess.engine
from utils import get_stockfish_engine

# Evaluate the position using Stockfish (default: SF14, 1 thread)
engine = get_stockfish_engine()
board = chess.Board(df.fen.iloc[109])
print(board.turn)
info = prob_win(engine, board, board.turn)
print(info)

board = chess.Board(df.fen.iloc[109])
print(board.turn)
info = prob_win(engine, board, ~board.turn)
print(info)

False


NameError: name 'prob_win' is not defined

In [ ]:
data = df.copy()
data["bin"] = data["move_ply"]
metrics = compute_metrics_by_bin(data)

# Use pandas qcut for quantile-based bins of move_ply
data["qbins"], qbin_edges = pd.qcut(data["bin"], q=30, duplicates="drop", retbins=True, labels=False)
metrics_qbin = compute_metrics_by_qbin(data, qbin_edges)

fig, (ax_raw, ax_qbin) = plt.subplots(1, 2, figsize=(10, 4))
plot_metrics(metrics, ax=ax_raw)
ax_raw.set_xlabel("Move Ply")
ax_raw.set_ylabel("Move Time (s)")
ax_raw.legend()

plot_metrics(metrics_qbin, ax=ax_qbin)
ax_qbin.set_xlabel("Move Ply (Quantile-binned)")
ax_qbin.set_ylabel("Move Time (s)")
ax_qbin.legend()

In [ ]:
#condition on time left
# data = df[df["time_left"] < 500]
# data = df
data = df.copy()
data["bin"] = data["player_time_left"]
plt.hist(data["bin"], bins = 30)
metrics = compute_metrics_by_bin(data)

# Use pandas qcut for quantile-based bins of move_ply
data["qbins"], qbin_edges = pd.qcut(data["bin"], q=20, duplicates="drop", retbins=True, labels=False)
metrics_qbin = compute_metrics_by_qbin(data, qbin_edges)

fig, (ax_raw, ax_qbin) = plt.subplots(1, 2, figsize=(10, 4))
plot_metrics(metrics, ax=ax_raw)
ax_raw.set_xlabel("Player Time Left (s)")
ax_raw.set_ylabel("Player Move Time (s)")
ax_raw.legend()

plot_metrics(metrics_qbin, ax=ax_qbin)
ax_qbin.set_xlabel("Player Time Left (Quantile-binned)")
ax_qbin.set_ylabel("Player Move Time (s)")
ax_qbin.legend()

In [ ]:
import numpy as np
from scipy.stats import linregress

# Use pandas qcut for quantile-based bins of move_ply
fig, ax = plt.subplots(1, 1, figsize=(10, 4))

import matplotlib as mpl

# Create a colormap that goes from light blue to dark blue
cmap = mpl.cm.Blues
norm = mpl.colors.Normalize(vmin=1, vmax=100)


plys = []
slopes = []
for ply in range(1, 100, 10): 
    data = df[df["move_ply"] == ply].copy()
    data["bin"] = data["total_time_left"]
    data["qbins"], qbin_edges = pd.qcut(data["bin"], q=5, duplicates="drop", retbins=True, labels=False)
    metrics_qbin = compute_metrics_by_qbin(data, qbin_edges)
    color = cmap(norm(ply))
    # plot_metrics accepts 'color' for lines/markers if you pass it (modify if needed)
    plot_metrics(metrics_qbin, ax=ax, color=color)

    if len(set(data["total_time_left"])) > 5: 
        # Perform linear regression: move_time ~ time_left
        x = data["total_time_left"]
        y = data["move_time"]

        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        time_left_move_time_slope = slope
        print("Slope of move_time ~ time_left:", time_left_move_time_slope)
        slopes.append(slope)
        plys.append(ply)
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
plt.plot(plys, slopes)

In [ ]:
import chess
import chess.engine
from utils import get_stockfish_engine
DEPTH_SHALLOW = 1
DEPTH_DEEP = 14

# Select a board position from the DataFrame
fen = df.board_position.iloc[0]
board = chess.Board(fen)

# Evaluate the position using Stockfish (default: SF14, 1 thread)
engine = get_stockfish_engine()

# Use a low depth for quick evaluation (e.g. depth=14)
info = engine.analyse(board, chess.engine.Limit(depth=14), info=chess.engine.INFO_ALL)
score_info = info["score"].white() if board.turn else info["score"].black()
print("Evaluation for position:")
print(fen)
print("Score:", score_info)
print("WDL:", score_info.wdl().wins / score_info.wdl().total())

In [ ]:
voc_consideration_set(engine, board)